### ML App (Credit Card Fraud Detection System) Part 1

#### Team members

- Alson Garbuja
- Kapil Bam
- Slok Baniya
- Nishan Parajuli
- Karma Dolma Gurung
- Sishir Kharel

In [1]:
pip install pandas numpy

Note: you may need to restart the kernel to use updated packages.


In [2]:
pip install scikit-learn

Note: you may need to restart the kernel to use updated packages.


In [3]:
"""
Imports

This cell has all the required imports from the third party libraries like pandas, numpy and sckit-learn (sklearn)
"""

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.linear_model import (
    LinearRegression,
    LogisticRegression,
    Perceptron,
    SGDClassifier
)
from sklearn.metrics import (
    mean_squared_error,
    r2_score,
    classification_report,
    confusion_matrix
)

In [11]:
"""
Data pre-processing

This cell contains the data pre-processing and cleaning of the data for the model to be fed
"""

df = pd.read_csv('datasets/eu_dataset/creditcard.csv')
df.dropna()

X_reg = df.drop(['Amount', 'Time', 'Class'], axis=1)
y_reg = df['Amount']

X_cls = df.drop(['Class', 'Time'], axis=1)
y_cls = df['Class']

In [5]:
"""
Data splitting

This cell uses the train_test_split function from sklearn package to split the dataset into training and testing sets.

The `stratify = y_cls` argument makes sure the extreme 99% to 1% normal to fraud data distribution in the original dataset is maintained 
in both training and testing splits of the data.
"""

X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(
    X_reg, y_reg, test_size=0.2, random_state=42
)

X_train_cls, X_test_cls, y_train_cls, y_test_cls = train_test_split(
    X_cls, y_cls, test_size=0.2, random_state=42, stratify=y_cls
)

In [6]:
"""
Data scaling

This cell holds the code to scale the data with mean of 0 and standard deviation of 1 so that numerically larger data doesnot dominate the
smaller data.
"""

scaler_reg = StandardScaler()
X_train_reg_scaled = scaler_reg.fit_transform(X_train_reg)
X_test_reg_scaled = scaler_reg.transform(X_test_reg)

scaler_cls = StandardScaler()
X_train_cls_scaled = scaler_cls.fit_transform(X_train_cls)
X_test_cls_scaled = scaler_cls.transform(X_test_cls)

In [7]:
"""
Training

This cell contains the code that uses Linear Regression model from sklearn package to train the model.
"""

print("=== REGRESSION TASKS (Predicting Transaction Amount) ===")

lin_reg = LinearRegression()
lin_reg.fit(X_train_reg_scaled, y_train_reg)
y_pred_lin = lin_reg.predict(X_test_reg_scaled)

print(f"Linear Regression : {mean_squared_error(y_test_reg, y_pred_lin):.2f}")
print(f"Linear Regression R2: {r2_score(y_test_reg, y_pred_lin):.2f}\n")

X_train_poly = X_train_reg_scaled[:10000]
y_train_poly = y_train_reg[:10000]
X_test_poly = X_test_reg_scaled[:2000]
y_test_poly = y_test_reg[:2000]

poly_features = PolynomialFeatures(degree=2, include_bias=False)
X_train_poly_feat = poly_features.fit_transform(X_train_poly)
X_test_poly_feat = poly_features.transform(X_test_poly)

poly_reg = LinearRegression()
poly_reg.fit(X_train_poly_feat, y_train_poly)
y_pred_poly = poly_reg.predict(X_test_poly_feat)

print(f"Polynomial Regression (Deg 2) MSE: {mean_squared_error(y_test_poly, y_pred_poly):.2f}")
print(f"Polynomial Regression (Deg 2) R2: {r2_score(y_test_poly, y_pred_poly):.2f}\n")

=== REGRESSION TASKS (Predicting Transaction Amount) ===
Linear Regression : 4333.59
Linear Regression R2: 0.92

Polynomial Regression (Deg 2) MSE: 627123.16
Polynomial Regression (Deg 2) R2: -11.00



In [8]:
"""
Logistic Regression model

This cell contains code that uses the Logistic Regression model to train and predict the fraud cases, and uses the classification_report 
function to display the report from the model.
"""

print("=== CLASSIFICATION TASKS (Detecting Fraud) ===")

log_reg = LogisticRegression(max_iter=1000, random_state=42)
log_reg.fit(X_train_cls_scaled, y_train_cls)
y_pred_log = log_reg.predict(X_test_cls_scaled)

print("--- Logistic Regression Report ---")
print(classification_report(y_test_cls, y_pred_log))

=== CLASSIFICATION TASKS (Detecting Fraud) ===
--- Logistic Regression Report ---
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     56864
           1       0.83      0.64      0.72        98

    accuracy                           1.00     56962
   macro avg       0.91      0.82      0.86     56962
weighted avg       1.00      1.00      1.00     56962



In [9]:
"""
Perceoptron model

This cell contains code that uses the Perceptron model to train and predict the fraud cases, and uses the classification_report function to
display the report from the model.
"""

perceptron = Perceptron(max_iter=1000, random_state=42)
perceptron.fit(X_train_cls_scaled, y_train_cls)
y_pred_prec = perceptron.predict(X_test_cls_scaled)

print("--- Perceptron Report ---")
print(classification_report(y_test_cls, y_pred_prec))

--- Perceptron Report ---
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     56864
           1       0.63      0.80      0.70        98

    accuracy                           1.00     56962
   macro avg       0.81      0.90      0.85     56962
weighted avg       1.00      1.00      1.00     56962



In [10]:
"""
Adaline model

This cell contains code that uses the SGDClassifier model with loss function set to 'squared_error' and learning_rate to 'constant' to 
create an adaline model to train and predict the fraud cases and also generate the report of the model.
"""

adaline = SGDClassifier(
    loss='squared_error',
    max_iter=1000,
    tol=1e-3,
    learning_rate='constant',
    eta0=0.001,
    random_state=42
)
adaline.fit(X_train_cls_scaled, y_train_cls)

y_pred_adaline_cont = adaline.predict(X_test_cls_scaled)
y_pred_adaline = np.where(y_pred_adaline_cont > 0.5, 1, 0)

print("--- Adaline (via SGD Squared Error) Report ---")
print(classification_report(y_test_cls, y_pred_adaline))

--- Adaline (via SGD Squared Error) Report ---
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     56864
           1       0.75      0.28      0.40        98

    accuracy                           1.00     56962
   macro avg       0.87      0.64      0.70     56962
weighted avg       1.00      1.00      1.00     56962

